In [25]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../../..").resolve()))

import pandas as pd
from src.utils.db import get_connection

import ast
import matplotlib.pyplot as plt
import plotly.express as px
import warnings
import datetime
from plotly.subplots import make_subplots
import plotly.graph_objects as go

warnings.filterwarnings('ignore')

# plt.rcParams['font.family'] = 'AppleGothic'    # Mac
plt.rcParams['font.family'] = 'Malgun Gothic' # Windows
plt.rcParams['axes.unicode_minus'] = False

# 1. 데이터 불러오기
conn = get_connection()

reviews_df = pd.read_sql(
    "SELECT * FROM steam_indie_reviews",
    conn
)

metadata_df = pd.read_sql(
    "SELECT * FROM steam_stratified_sample",
    conn
)

conn.close()


reviews_df['author_playtime_forever'] = reviews_df['author_playtime_forever'].astype('int64')
reviews_df['author_playtime_at_review'] = reviews_df['author_playtime_at_review'].astype('int64')
metadata_df['owners_lower'] = metadata_df['owners_lower'].astype('int64')
metadata_df['positive'] = metadata_df['positive'].astype('int64')
metadata_df['negative'] = metadata_df['negative'].astype('int64')
metadata_df['total_reviews'] = metadata_df['total_reviews'].astype('int64')
reviews_df['timestamp_created'] = reviews_df['timestamp_created'].astype('int64')
reviews_df['timestamp_created'] = reviews_df['timestamp_created'].apply(lambda x: datetime.datetime.fromtimestamp(x))
reviews_df['timestamp_created'] = reviews_df['timestamp_created'].dt.tz_localize('UTC').dt.tz_convert('Asia/Seoul')
reviews_df['timestamp_updated'] = reviews_df['timestamp_updated'].astype('int64')
reviews_df['timestamp_updated'] = reviews_df['timestamp_updated'].apply(lambda x: datetime.datetime.fromtimestamp(x))
reviews_df['timestamp_updated'] = reviews_df['timestamp_updated'].dt.tz_localize('UTC').dt.tz_convert('Asia/Seoul')


In [2]:
success_df = metadata_df[metadata_df['stratum'].apply(lambda x: 'large_high' in x)]
fail_df = metadata_df[metadata_df['stratum'].apply(lambda x: 'small_high' in x)]

- 흥행 성공과 실패의 기준은?
- 음... 일단 large_high와 small_high를 써보자

In [3]:
# 장르 조합 추출
def get_genre_combo_no_indie(genre_str):
    try:
        genres = ast.literal_eval(genre_str)
        filtered = [g for g in genres if g != 'Indie']
        return ", ".join(sorted(filtered)) if filtered else None
    except:
        return None

success_df['genre_combo_clean'] = success_df['genres'].apply(get_genre_combo_no_indie)
fail_df['genre_combo_clean'] = fail_df['genres'].apply(get_genre_combo_no_indie)
success_df = success_df.dropna(subset=['genre_combo_clean'])
fail_df = fail_df.dropna(subset=['genre_combo_clean'])

In [4]:
# 가장 많은 장르 조합 15개 선정 및 해당 게임 외 필터링
success_top_combos = success_df['genre_combo_clean'].value_counts().nlargest(15).index
fail_top_combos = fail_df['genre_combo_clean'].value_counts().nlargest(15).index

df_top_success = success_df[success_df['genre_combo_clean'].isin(success_top_combos)].copy()
df_top_fail = fail_df[fail_df['genre_combo_clean'].isin(fail_top_combos)].copy()

# 장르 조합별 판매량 하한선의 중앙값으로 장르 조합 순서 정렬
success_combo_order = df_top_success.groupby('genre_combo_clean')['owners_lower'].median().sort_values(ascending=False).index.tolist()
fail_combo_order = df_top_fail.groupby('genre_combo_clean')['owners_lower'].median().sort_values(ascending=False).index.tolist()

In [5]:
print(f"large_high의 장르 조합: {len(success_combo_order)}\nsmall_high의 장르 조합: {len(fail_combo_order)}")

large_high의 장르 조합: 15
small_high의 장르 조합: 15


In [6]:
# large_high와 small_high의 판매량 하한선 중앙값 기준 장르 조합 순위 TOP 15
df_combos = pd.DataFrame({
    'large_high' : success_combo_order,
    'small_high' : fail_combo_order
})

df_combos

,large_high,small_high
0,"Action, Adventure, RPG","Action, Adventure"
1,"Adventure, RPG, Simulation","Action, Adventure, Casual, RPG, Simulation"
2,"Adventure, Casual, RPG, Simulation","Action, Adventure, Casual, Racing, Simulation,..."
3,"Casual, Free To Play, Strategy","Action, Adventure, RPG"
4,"Casual, Simulation","Action, Casual, Free To Play, Sports"
5,"Action, Casual, Simulation","Action, Early Access, Simulation"
6,Adventure,Adventure
7,"Action, Adventure, Casual, Simulation, Strategy","Adventure, Casual"
8,Action,"Adventure, Early Access, Simulation"
9,"Action, Adventure","Adventure, RPG, Strategy"


In [26]:
# 1. 시계열 데이터 변환 및 연월(YYYY-MM) 컬럼 생성
# utc=True를 설정하면 다양한 시간대 형식을 안전하게 처리할 수 있습니다.
reviews_df['dt_created'] = pd.to_datetime(reviews_df['timestamp_created'], utc=True)
reviews_df['year_month'] = reviews_df['dt_created'].dt.strftime('%Y-%m')

# 2. 메타데이터와 병합하여 stratum 정보 가져오기
df_trend = pd.merge(
    reviews_df[['recommendationid', 'appid', 'year_month']], 
    metadata_df[['appid', 'stratum']], 
    on='appid'
)

In [27]:
# 3. 대상 그룹(large_high, small_high) 필터링
target_stratums = ['large_high', 'small_high']
df_filtered = df_trend[df_trend['stratum'].isin(target_stratums)]

# 4. 그룹화 및 리뷰 수 카운트
# stratum과 year_month를 기준으로 그룹화합니다.
trend_counts = df_filtered.groupby(['stratum', 'year_month']).size().reset_index(name='review_count')

# 날짜 순서대로 정렬 (YYYY-MM 문자열은 사전순 정렬이 곧 시간순 정렬입니다)
trend_counts = trend_counts.sort_values('year_month')

In [28]:
# 5. Plotly 시각화
fig_trend = px.line(
    trend_counts, 
    x='year_month', 
    y='review_count', 
    color='stratum',
    markers=True,
    title='Stratum별 연월 리뷰 등록 수 추이 (large_high vs small_high)',
    labels={'year_month': '등록 연월', 'review_count': '리뷰 수', 'stratum': '그룹'},
    color_discrete_map={'large_high': '#636EFA', 'small_high': '#EF553B'} # 색상 지정
)

fig_trend.update_layout(
    xaxis_tickangle=-45, # X축 라벨 기울여서 가독성 확보
    template="plotly_white",
    hovermode="x unified"
)

fig_trend.show()

In [30]:
# 1. 데이터 준비 및 출시연도 추출
reviews_df['dt_created'] = pd.to_datetime(reviews_df['timestamp_created'], utc=True)
reviews_df['year_month'] = reviews_df['dt_created'].dt.strftime('%Y-%m')

# 메타데이터 병합 (출시일과 장르 포함)
df_full = pd.merge(
    reviews_df[['appid', 'year_month']], 
    metadata_df[['appid', 'stratum', 'release_date', 'genres']], 
    on='appid'
)

# 출시연도 컬럼 생성 (YYYY)
df_full['release_year'] = pd.to_datetime(df_full['release_date']).dt.year.astype(str)

In [31]:
# 2. [관점 A] 출시 연도별 리뷰 등록 추이 (large_high 그룹 대상)
df_large = df_full[df_full['stratum'] == 'large_high']
trend_by_release = df_large.groupby(['release_year', 'year_month']).size().reset_index(name='count')

fig_release = px.line(
    trend_by_release, x='year_month', y='count', color='release_year',
    title='[Large High] 출시 연도별 게임들의 리뷰 등록 추이',
    labels={'year_month': '리뷰 등록 연월', 'count': '리뷰 수', 'release_year': '게임 출시년도'}
)
fig_release.update_layout(xaxis_tickangle=-45, template="plotly_white")
fig_release.show()

In [32]:
# 3. [관점 B] 주요 장르별 리뷰 등록 추이
target_genres = ["Action", "RPG", "Simulation", "Strategy"]
df_full['genres_list'] = df_full['genres'].apply(ast.literal_eval)
df_genre_trend = df_full.explode('genres_list')
df_genre_trend = df_genre_trend[df_genre_trend['genres_list'].isin(target_genres)]

trend_by_genre = df_genre_trend.groupby(['genres_list', 'year_month']).size().reset_index(name='count')

fig_genre_trend = px.line(
    trend_by_genre, x='year_month', y='count', color='genres_list',
    title='주요 장르별 시간에 따른 리뷰 등록 수 변화',
    labels={'year_month': '리뷰 등록 연월', 'count': '리뷰 수', 'genres_list': '장르'}
)
fig_genre_trend.update_layout(xaxis_tickangle=-45, template="plotly_white")
fig_genre_trend.show()

In [33]:
# 1. 데이터 준비
reviews_df['dt_created'] = pd.to_datetime(reviews_df['timestamp_created'], utc=True)
reviews_df['year_month'] = reviews_df['dt_created'].dt.strftime('%Y-%m')

df_full = pd.merge(
    reviews_df[['appid', 'year_month']], 
    metadata_df[['appid', 'stratum', 'release_date', 'genres']], 
    on='appid'
)

# 출시연도 추출 (데이터가 충분한 최근 연도 위주로 필터링 가능)
df_full['release_year'] = pd.to_datetime(df_full['release_date']).dt.year.astype(str)
release_years = sorted(df_full['release_year'].unique())[-3:] # 최근 3개년 예시

# 주요 4대 장르 설정
target_genres = ["Action", "RPG", "Simulation", "Strategy"]
df_full['genres_list'] = df_full['genres'].apply(ast.literal_eval)
df_comp = df_full.explode('genres_list')
df_comp = df_comp[df_comp['genres_list'].isin(target_genres)]

In [34]:
# 2. 서브플롯 생성 (행: 출시연도, 열: 1)
fig = make_subplots(
    rows=len(release_years), cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=[f"{yr}년 출시 게임들의 장르별 리뷰 추이" for yr in release_years]
)

colors = px.colors.qualitative.Safe

# 3. 데이터 루프 및 추가
for r_idx, year in enumerate(release_years):
    year_df = df_comp[df_comp['release_year'] == year]
    
    for g_idx, genre in enumerate(target_genres):
        genre_df = year_df[year_df['genres_list'] == genre]
        trend = genre_df.groupby('year_month').size().reset_index(name='count').sort_values('year_month')
        
        fig.add_trace(
            go.Scatter(
                x=trend['year_month'], y=trend['count'],
                name=genre, mode='lines+markers',
                line=dict(color=colors[g_idx]),
                legendgroup=genre,
                showlegend=(r_idx == 0) # 범례는 한 번만 표시
            ),
            row=r_idx + 1, col=1
        )
        

In [35]:
# 4. 레이아웃 정리
fig.update_layout(
    height=300 * len(release_years),
    template="plotly_white",
    title_text="출시 시점 및 장르에 따른 리뷰 활동성 분석",
    hovermode="x unified"
)
fig.update_xaxes(tickangle=-45)

fig.show()

In [36]:
# 1. 데이터 전처리 및 Stratum 병합
reviews_df['dt_created'] = pd.to_datetime(reviews_df['timestamp_created'], utc=True)
reviews_df['year_month'] = reviews_df['dt_created'].dt.strftime('%Y-%m')

# 리뷰와 메타데이터 병합
df_full = pd.merge(
    reviews_df[['appid', 'year_month']], 
    metadata_df[['appid', 'stratum', 'release_date', 'genres']], 
    on='appid'
)

# 출시연도 및 장르 리스트화
df_full['release_year'] = pd.to_datetime(df_full['release_date']).dt.year.astype(str)
df_full['genres_list'] = df_full['genres'].apply(ast.literal_eval)

# 분석 대상 설정
target_stratums = ['large_high', 'small_high']
target_genres = ["Action", "RPG", "Simulation", "Strategy"]
target_years = sorted(df_full['release_year'].unique())[-3:] # 최근 3개년

In [37]:
# 2. Stratum별로 루프를 돌며 차트 생성
for target_stratum in target_stratums:
    # 해당 Stratum 데이터만 필터링
    df_stratum = df_full[df_full['stratum'] == target_stratum].explode('genres_list')
    df_stratum = df_stratum[df_stratum['genres_list'].isin(target_genres)]
    
    # 서브플롯 생성 (행: 출시연도)
    fig = make_subplots(
        rows=len(target_years), cols=1,
        shared_xaxes=True,
        vertical_spacing=0.07,
        subplot_titles=[f"[{target_stratum}] {yr}년 출시 게임의 장르별 추이" for yr in target_years]
    )
    
    colors = px.colors.qualitative.Safe
    
    for r_idx, year in enumerate(target_years):
        year_df = df_stratum[df_stratum['release_year'] == year]
        
        for g_idx, genre in enumerate(target_genres):
            genre_df = year_df[year_df['genres_list'] == genre]
            trend = genre_df.groupby('year_month').size().reset_index(name='count').sort_values('year_month')
            
            fig.add_trace(
                go.Scatter(
                    x=trend['year_month'], y=trend['count'],
                    name=genre, mode='lines+markers',
                    line=dict(color=colors[g_idx % len(colors)]),
                    legendgroup=genre,
                    showlegend=(r_idx == 0)
                ),
                row=r_idx + 1, col=1
            )

    fig.update_layout(
        height=350 * len(target_years),
        title_text=f"{target_stratum} 그룹의 출시 연도 및 장르별 리뷰 활동성 분석",
        template="plotly_white",
        hovermode="x unified"
    )
    fig.show()